# Week 8: Listing Summarization

This notebook reviews the listing summarizer on a set of real MLS listings.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.real_estate_nlp.answerability_checker import AnswerabilityChecker
from src.real_estate_nlp.query_parser import QueryParser
from src.real_estate_nlp.schema_validator import SchemaValidator

pd.set_option("display.max_colwidth", None)

---

## 1. Artifacts Loading


In [2]:
source = pd.read_csv('../data/processed/listing_summary_eval_source.csv')
labels = json.loads(Path('../data/processed/listing_summary_eval_labels.json').read_text())['items']
dev_results = json.loads(Path('../data/processed/listing_summary_dev_results.json').read_text())
test_results = json.loads(Path('../data/processed/listing_summary_eval_results.json').read_text())
test_rows = pd.DataFrame(test_results['rows'])
self_review = pd.read_csv('../data/processed/listing_summary_self_review.csv')

source['listing_id'] = source['listing_id'].astype(str)


---

## 2. Evaluation Sample

The sample combines complete listings with sparse or incomplete records. This checks both normal summary quality and the fallback behavior needed for real MLS data.


In [3]:
sample_profile = source.groupby('split').agg(
    listings=('listing_id', 'size'),
    median_remark_characters=('remarks', lambda values: values.fillna('').str.len().median()),
    missing_city=('city', lambda values: values.isna().sum()),
    missing_remarks=('remarks', lambda values: values.isna().sum()),
)
sample_profile


,listings,median_remark_characters,missing_city,missing_remarks
split,,,,
dev,20,1152.5,1,2
test,30,998.5,0,1


In [4]:
source[['listing_id', 'split', 'city', 'price', 'beds', 'baths', 'remarks']].head(3)


,listing_id,split,city,price,beds,baths,remarks
0,1160075181,dev,Riverside,680000,4.0,2.0,"Welcome to this incredible Riverside property offering the perfect blend of space, comfort, and opportunity. From the moment you arrive, the circular driveway, RV parking, mature landscaping, and oversized lot set the tone for a home that truly stands out.\r\r\n\r\r\nInside, you’ll find 4 spacious bedrooms, 2 beautifully remodeled bathrooms, and approximately 1,636 sq. ft. of comfortable living space designed for both everyday living and entertaining. A dedicated dining room creates the perfect setting for gatherings and special meals, while an additional bonus, 5th room provides incredible flexibility for a home office, gym, playroom, guest space, or whatever best fits your lifestyle.\r\r\n\r\r\nThe upgraded kitchen features shaker-style cabinetry, butcher block countertops, and stainless steel appliances, blending warmth with modern style. Laminate flooring, updated finishes, central heating and air, indoor laundry, and a cozy fireplace all add to the home’s comfort and functionality.\r\r\n\r\r\nStep outside and discover one of the property’s true highlights — a spacious backyard complete with a large covered patio, ideal for entertaining, relaxing, or enjoying year-round outdoor living. With plenty of room to garden, expand, or create your dream backyard setup, the possibilities are endless.\r\r\n\r\r\nThe detached 2-car garage also offers excellent ADU potential for future rental income or multigenerational living (buyer to verify with city). Conveniently located near schools, shopping, parks, freeways, and just minutes from the Galleria at Tyler, this home delivers both everyday convenience and long-term potential.\r\r\n\r\r\nHomes offering this much space, parking, flexibility, and outdoor living rarely come available — this is one you truly need to experience in person."
1,1168506861,dev,Barstow,110000,3.0,2.0,Investors wanted!! Would be a great rental. Seller had started with some updates and finishing permits and just never continued. This can be your gain$$ Come take a look
2,1159850513,dev,San Jose,1798000,3.0,2.0,"Located in the heart of coveted Willow Glen, this 3 bed 2-bath bungalow blends modern luxury with timeless California charm. Completely remodeled throughout-this stunning home features a newly renovated chefs kitchen with gleaming quartz countertops, new cabinetry,gas cooktop, and an inviting dining area.Brand-new luxury plank flooring and fresh designer paint.The spacious floor plan offers three generous bedrooms, including a serene primary retreat with a walk-in closet. Situated on an expansive 7,374 square foot lot, the property also includes a detached approximately 650 sq. ft. ADU complete with its own kitchenette and full bath, offering excellent income potential or flexible guest accommodations. A separate 200 sq. ft. office provides the ideal work-from-home setup or creative studio space. Step outside to a peaceful backyard oasis featuring a lush lawn, patio area, and mature fruit trees that create a private and inviting setting for relaxing or entertaining. Rarely does a property offer this level of versatility, character, and modern functionality in one of San Joses most beloved neighborhoods. Just moments from downtown Willow Glens vibrant shops, restaurants, and year-round community events, this is a rare opportunity to own a truly unique and exceptional property."


---

## 3. Evaluation

ROUGE-L compares each generated summary with an independently written reference summary.

Fact coverage checks whether the MLS facts and manually selected feature terms in the evaluation label appear in the output.


In [5]:
metrics = ['split', 'listings_evaluated', 'rouge_l', 'fact_coverage']
evaluation_summary = pd.DataFrame([
    {key: dev_results[key] for key in metrics},
    {key: test_results[key] for key in metrics},
])
evaluation_summary


,split,listings_evaluated,rouge_l,fact_coverage
0,dev,20,0.463941,0.815000
1,test,30,0.456898,0.733333


---

## 4. Summary Examples

These examples make it easier to check whether the short output preserves the most useful facts without turning into a rewritten listing description.


In [6]:
examples = source.merge(
    test_rows[['listing_id', 'reference_summary', 'summary', 'rouge_l', 'fact_coverage']],
    on='listing_id',
)
examples[['city', 'price', 'beds', 'baths', 'remarks', 'reference_summary', 'summary', 'rouge_l',
          'fact_coverage']].sample(
    5, random_state=0
)


,city,price,beds,baths,remarks,reference_summary,summary,rouge_l,fact_coverage
2,Rancho Dominguez,599800,2.0,1.0,"Affordable entry starter home opportunity in the East Rancho Dominguez/East Compton area! This property is move in ready and features a newly remodeled modern kitchen, fresh interior paint throughout, and new waterproof laminate wood flooring. Large lot offers excellent potential for future expansion, including the possibility to add additional living area square footage or add an ADU (Accessory Dwelling Unit). Walking distance to Whaley Middle School, convenient access to the 710, 105, 91, and 405 freeways.","At $599,800, this 2-bed, 1-bath Rancho Dominguez home pairs a remodeled kitchen with a large lot.","This 2-bed, 1-bath listing in Rancho Dominguez is listed at $599,800. Highlights include an accessory dwelling unit and a remodeled interior.",0.418605,0.666667
28,Bakersfield,295000,3.0,2.0,"Charming and move-in ready, this remodeled 3-bedroom, 2-bath home features a bright, modern interior with updated kitchen and bathrooms, sleek fixtures, and neutral tones throughout. The functional layout includes comfortable living spaces and well-sized bedrooms.\r\r\n\r\r\nThe spacious backyard provides room for outdoor enjoyment, with a recently installed gate for added privacy and convenience. A front fence enhances curb appeal.\r\r\n\r\r\nConveniently located near parks, schools, shopping, and other local amenities.","This $295,000 3-bed, 2-bath Bakersfield home has a remodeled kitchen and backyard.","This 3-bed, 2-bath listing in Bakersfield is listed at $295,000. Highlights include a remodeled interior and a backyard.",0.555556,0.833333
13,Highland,809990,4.0,3.0,"NEW CONSTRUCTION - SINGLE-FAMILY HOMES - FORMER MODEL HOME! Vista Verde at Mediterra in Highland is nestled against the base of the foothills and features a beautiful pool and park. THIS FORMER MODEL HOME RESIDENCE 2,239 HAS IT ALL – welcoming covered porch and beautiful entry, designer upgrades, a main level bedroom and full bath, open, expansive Great Room, Kitchen and Dining Space designed for gathering and entertaining and a gourmet Kitchen with stylish cabinetry and counter tops, kitchen island and stainless-steel appliances. Take the festivities outside and enjoy the nicely-sized landscaped back yard, or retreat to the amazing, oversized loft space upstairs for that perfect movie night, or studying or lounging. The luxurious Primary Suite is a welcome refuge from the day, while the Primary Bath offers a spa-like space to unwind with an oversized shower and separate tub, dual sink vanity with stone counter tops and large walk-in closet. Additional spacious bedrooms and dedicated laundry room with extra storage complements this well-appointed home. Other amazing features include ""Americas Smart Home Technology"" for Home Automation at your fingertips, LED recessed lighting and much more.","This $809,990 4-bed, 3-bath Highland former model home includes a landscaped backyard.","This 4-bed, 3-bath listing in Highland is listed at $809,990. Highlights include a pool and new construction.",0.457143,0.666667
10,Brentwood,894999,5.0,3.0,"Nestled in the quiet Orchard Park neighborhood of Brentwood, this stunning 5 bedroom, 2.5 bath home offers the perfect blend of space and tranquility. Inside, you'll discover a thoughtfully designed layout with ample natural light, an open floor plan along with newly constructed shelving, a custom entertainment center and dry bar. Outside, the covered gazebo with electrical and refrigerator make this home perfect for hosting and entertaining, inside and out! Don't miss this incredible opportunity to own your own little piece of Brentwood!","This $894,999 5-bed, 3-bath Brentwood home has an open floor plan and a covered gazebo for outdoor entertaining.","This 5-bed, 3-bath listing in Brentwood is listed at $894,999. Highlights include an open floor plan and new construction.",0.511628,0.833333
26,Anza,365000,4.0,2.0,"WELCOME

---

## 5. Answerability Checks

Before a query runs, the checker distinguishes valid searches from unsupported questions and invalid constraints.

After execution, it turns empty or unusable result sets into a clear response.


In [7]:
checker = AnswerabilityChecker(QueryParser(), SchemaValidator())

requests = [
    'Show me 3-bed homes in Irvine under $900,000',
    'Show homes in Irvine under $1',
    'What does DOM mean?',
    'How do I bake bread?',
]

pre_query = pd.DataFrame([
    {'request': request, 'answerable': allowed, 'message': message}
    for request in requests
    for allowed, message in [checker.check_pre_query(request)]
])
pre_query


,request,answerable,message
0,"Show me 3-bed homes in Irvine under $900,000",True,Query is answerable.
1,Show homes in Irvine under $1,False,Query references invalid data: price_max=1 is outside the supported range
2,What does DOM mean?,False,"This is a real estate question, but it cannot be answered by the current listing search."
3,How do I bake bread?,False,This doesn't appear to be a real estate listing search.


In [8]:
result_sets = {
    'No matches': pd.DataFrame(),
    'No usable values': pd.DataFrame({'price': [None]}),
    'Listings returned': pd.DataFrame({'price': [950000]}),
}

post_query = pd.DataFrame([
    {'result_state': name, 'answerable': allowed, 'message': message}
    for name, results in result_sets.items()
    for allowed, message in [checker.check_post_query(results)]
])
post_query


,result_state,answerable,message
0,No matches,False,No listings match your criteria.
1,No usable values,False,Query returned no meaningful listing data.
2,Listings returned,True,Results found.
